In [2]:
import mlflow 
mlflow.autolog(silent=True)

import mlflow
import optuna
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import r2_score, root_mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from tqdm import tqdm
import os

from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt

In [3]:
mlflow.set_experiment("Reco Agricole")
mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI"))

In [4]:
knn_df = pd.read_csv("../data/processed/knn_merged.csv")
knn_df.dtypes

yield (t/ha)           float64
rain (mm)              float64
temp (C)               float64
item                    object
Year                     int64
                        ...   
Area_Ukraine              bool
Area_United Kingdom       bool
Area_Uruguay              bool
Area_Zambia               bool
Area_Zimbabwe             bool
Length: 113, dtype: object

In [ ]:
def optimize_xgboost(df, target_col, num_folds=5, n_trials=20, params={}):
    train = df[df[target_col].notna()]
    test = df[df[target_col].isna()]

    X_train = train.drop(target_col, axis=1)
    y_train = train[target_col]

    # Define the Optuna objective function
    def objective(trial):
        # Suggest hyperparameters
        optuna_params = {
        # Core parameters
        'objective': 'reg:squarederror',
        'eval_metric': 'auc',
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.3, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 50, 1000),
        'min_child_weight': trial.suggest_float('min_child_weight', 1e-3, 10, log=True),
        'n_jobs': -1
    }

        # Perform k-fold cross-validation
        kf = KFold(n_splits=num_folds, shuffle=True, random_state=42)
        scores = []
        for train_index, val_index in kf.split(X_train, y_train):
            X_train_kf, X_val_kf = X_train.iloc[train_index], X_train.iloc[val_index]
            y_train_kf, y_val_kf = y_train.iloc[train_index], y_train.iloc[val_index]

            model = XGBRegressor(**optuna_params)
            model.fit(X_train_kf, y_train_kf)
            score = root_mean_squared_error(y_val_kf, model.predict(X_val_kf))
            scores.append(score)

        return sum(scores) / len(scores)

    # Run Optuna optimization
    study = optuna.create_study(direction='minimize')
    if params:
        study.enqueue_trial(params)
    study.optimize(objective, n_trials=n_trials)

    # Print the best parameters and score
    print(f"Best trial: {study.best_trial.value}")
    print(f"Best params: {study.best_params}")

    # Train the final model with the best parameters
    best_params = study.best_params

    # Print k-fold scores with the best model
    kf = KFold(n_splits=num_folds, shuffle=True, random_state=42)
    scores = []
    for i, (train_index, val_index) in enumerate(kf.split(X_train, y_train, y_train)):
        X_train_kf, X_val_kf = X_train.iloc[train_index], X_train.iloc[val_index]
        y_train_kf, y_val_kf = y_train.iloc[train_index], y_train.iloc[val_index]

        model = XGBRegressor(**best_params)
        model.fit(X_train_kf, y_train_kf)
        score = root_mean_squared_error(y_val_kf, model.predict(X_val_kf))
        scores.append(score)
        print(f"Fold {i}, RMSE: {score}")

        mae = mean_absolute_error(y_val_kf, model.predict(X_val_kf))
        mape = mean_absolute_percentage_error(y_val_kf, model.predict(X_val_kf))
        r2 = r2_score(y_val_kf, model.predict(X_val_kf))

        mlflow.log_metric("rmse", score)
        mlflow.log_metric("mae", mae)
        mlflow.log_metric("mape", mape)
        mlflow.log_metric("r2", r2)

    print(f"Mean RMSE: {sum(scores) / len(scores)}")

    return study.best_params

In [5]:
for crop in knn_df["item"].unique():
    print("Crop : ", crop)
    with mlflow.start_run(experiment_id=1, run_name="xgboost_finetuning", tags={
            "phase": "finetuning",
            "model_family": "xgboost",
        }):
        df = knn_df[knn_df["item"] == crop].drop("item", axis=1)
        best_params = optimize_xgboost(df, "yield (t/ha)")
    print(best_params)

[I 2026-03-12 19:31:54,284] A new study created in memory with name: no-name-7b9d8384-82df-4cee-abf4-e75ae91fec6a


Crop :  Wheat


[I 2026-03-12 19:32:22,310] Trial 0 finished with value: 0.24135187551326692 and parameters: {'max_depth': 6, 'learning_rate': 0.10656719188502974, 'n_estimators': 611, 'min_child_weight': 0.062066096848362354}. Best is trial 0 with value: 0.24135187551326692.
[I 2026-03-12 19:32:46,963] Trial 1 finished with value: 1.1507528561555893 and parameters: {'max_depth': 9, 'learning_rate': 0.0036889634321628038, 'n_estimators': 145, 'min_child_weight': 0.012237429484164508}. Best is trial 0 with value: 0.24135187551326692.
[I 2026-03-12 19:33:16,743] Trial 2 finished with value: 0.24547707610592698 and parameters: {'max_depth': 9, 'learning_rate': 0.032818050261722985, 'n_estimators': 568, 'min_child_weight': 8.571286579962274}. Best is trial 0 with value: 0.24135187551326692.
[I 2026-03-12 19:33:51,595] Trial 3 finished with value: 0.23432293446551267 and parameters: {'max_depth': 9, 'learning_rate': 0.036127181952545176, 'n_estimators': 797, 'min_child_weight': 0.07092527531262588}. Best i

Best trial: 0.22599680248991066
Best params: {'max_depth': 11, 'learning_rate': 0.05354753457601451, 'n_estimators': 646, 'min_child_weight': 2.718311331068721}
Fold 0, RMSE: 0.2596425874117341


/home/bob/Documents/Dev/openclassrooms/reco-agricole/.venv/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [19:47:04] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "verbose" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


...

Fold 2, RMSE: 0.2849861683168236


/home/bob/Documents/Dev/openclassrooms/reco-agricole/.venv/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [20:16:29] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "verbose" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Fold 3, RMSE: 0.28699334638327156


/home/bob/Documents/Dev/openclassrooms/reco-agricole/.venv/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [20:16:38] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "verbose" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Fold 4, RMSE: 0.3122842998472582
Mean RMSE: 0.2829143373844485
🏃 View run xgboost_finetuning at: http://localhost:5000/#/experiments/1/runs/17aaec49f8034410a9eb5887fb5a849d
🧪 View experiment at: http://localhost:5000/#/experiments/1
{'max_depth': 8, 'learning_rate': 0.07181191872536907, 'n_estimators': 798, 'min_child_weight': 0.003637324617583913}


In [6]:
best_params_Wheat = {'max_depth': 11, 'learning_rate': 0.05354753457601451, 'n_estimators': 646, 'min_child_weight': 2.718311331068721}
best_params_Maize = {'max_depth': 9, 'learning_rate': 0.14338257391347994, 'n_estimators': 829, 'min_child_weight': 1.6986706925579842}
best_params_Rice = {'max_depth': 8, 'learning_rate': 0.07181191872536907, 'n_estimators': 798, 'min_child_weight': 0.003637324617583913}

In [10]:
with mlflow.start_run(experiment_id=1, run_name="xgboost_finetuned", tags={
            "phase": "final",
            "model_family": "xgboost",
            "dataset": "Wheat"
        }):
    df = knn_df[knn_df["item"] == "Wheat"].drop("item", axis=1)
    target_col = "yield (t/ha)"

    model = XGBRegressor(**best_params_Wheat)
    model.fit(df.drop(target_col, axis=1), df[target_col])

    mlflow.xgboost.log_model(model, "model-Wheat")

2026/03/12 21:13:20 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run xgboost_finetuned at: http://localhost:5000/#/experiments/1/runs/c3f210f7c3e943f8a53fd3f4ebc9778b
🧪 View experiment at: http://localhost:5000/#/experiments/1


In [11]:
with mlflow.start_run(experiment_id=1, run_name="xgboost_finetuned", tags={
            "phase": "final",
            "model_family": "xgboost",
            "dataset": "Rice"
        }):
    df = knn_df[knn_df["item"] == "Rice"].drop("item", axis=1)
    target_col = "yield (t/ha)"

    model = XGBRegressor(**best_params_Rice)
    model.fit(df.drop(target_col, axis=1), df[target_col])

    mlflow.xgboost.log_model(model, "model-Rice")

2026/03/12 21:13:35 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run xgboost_finetuned at: http://localhost:5000/#/experiments/1/runs/902accdf172a47e0a4a9485951455e59
🧪 View experiment at: http://localhost:5000/#/experiments/1


In [12]:
with mlflow.start_run(experiment_id=1, run_name="xgboost_finetuned", tags={
            "phase": "final",
            "model_family": "xgboost",
            "dataset": "Maize"
        }):
    df = knn_df[knn_df["item"] == "Maize"].drop("item", axis=1)
    target_col = "yield (t/ha)"

    model = XGBRegressor(**best_params_Maize)
    model.fit(df.drop(target_col, axis=1), df[target_col])

    mlflow.xgboost.log_model(model, "model-Maize")

2026/03/12 21:13:48 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run xgboost_finetuned at: http://localhost:5000/#/experiments/1/runs/afaf5323376b40128dea23f89fc9dff9
🧪 View experiment at: http://localhost:5000/#/experiments/1


In [19]:
df = knn_df[knn_df["item"] == "Maize"].drop("item", axis=1)
model.predict(df.drop("yield (t/ha)", axis=1).sample(1))

array([1.7060776], dtype=float32)

In [8]:
for crop in knn_df["item"].unique():    
    model = mlflow.xgboost.load_model(f"models:/model-{crop}/None")
    model.save_model(f"../src/models/{crop.lower()}_model.json")

KeyboardInterrupt: 